# Notebook 03 — Static Synthetic Data Generation

This notebook:
- Loads the `target_cohorts.json` produced by Notebook 02
- Trains a **GaussianCopulaSynthesizer** (SDV) on the full training data
- Generates targeted synthetic samples oversampling the identified weak cohorts
- Evaluates generation quality (KS statistics, distribution plots)
- Saves the synthesizer and `synthetic_samples.csv`

In [ ]:
import sys, warnings, json
warnings.filterwarnings('ignore')
sys.path.insert(0, '.')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from generator_utils import (
    train_generator,
    sample_targeted_synthetic_data,
    evaluate_synthetic_quality,
    save_generator,
)

%matplotlib inline
print('Imports OK')

## 1. Load Artefacts from Notebooks 01 & 02

In [ ]:
# Raw training data (un-scaled — required by SDV generator)
X_train = pd.read_csv('artifacts/X_train_clean.csv')
y_train = pd.read_csv('artifacts/y_train_clean.csv').squeeze()

# Full training DataFrame including target (for the generator)
train_full = pd.concat([X_train, y_train.rename('is_fraud')], axis=1)

print(f'Training samples : {len(train_full)}')
print(f'Columns          : {list(train_full.columns)}')
print(f'Minority class   : {y_train.sum()} fraud_cases ({y_train.mean()*100:.1f}%)')

# Target cohorts identified by the error analysis notebook
with open('artifacts/target_cohorts.json') as f:
    target_cohorts = json.load(f)

print(f'\nLoaded {len(target_cohorts)} target cohorts:')
for c in target_cohorts:
    print(f"  {c['name']:25s}  recall={c['recall']:.3f}")

## 2. Train Gaussian Copula Synthesizer

In [ ]:
# Train on the full training DataFrame
generator = train_generator(
    real_data=train_full,
    target_col='is_fraud',
    generator_type='copula',
)
print('Generator ready.')

## 3. Generate Targeted Synthetic Samples

In [ ]:
# Build raw conditions for each cohort:
def build_filter_conditions(cohorts, train_df):
    """Convert cohort metadata into simple {col: value} equality filters
    or use continuous slicing for range-based cohorts."""
    enriched = []
    for c in cohorts:
        if c['cohort_type'] == 'range':
            col = c['group_col']
            conds = c['conditions']
            lo = conds.get(f'{col}_low')
            hi = conds.get(f'{col}_high')
            if lo is not None and hi is not None:
                mask = (train_df[col] > lo) & (train_df[col] <= hi) # match pd.Interval behavior
            else:
                mask = pd.Series([True]*len(train_df), index=train_df.index)
            enriched.append({**c, '_mask': mask})
        else:
            enriched.append({**c, '_mask': None})
    return enriched

enriched_cohorts = build_filter_conditions(target_cohorts, train_full)

# Sample for each cohort separately to honour range filters
all_synthetic_parts = []
target_col_name = y_train.name

for cohort in enriched_cohorts:
    mask = cohort.get('_mask')
    if mask is not None:
        cohort_data = train_full[mask].copy()
    else:
        col = cohort['group_col']
        val = cohort['group_val']
        # attempt to cast val back to numeric if column is numeric
        if pd.api.types.is_numeric_dtype(train_full[col]):
            try: val = float(val)
            except: pass
        cohort_data = train_full[train_full[col] == val].copy()

    if len(cohort_data) < 5:
        print(f"[!] Cohort '{cohort['name']}' has < 5 rows — skipping.")
        continue

    # Number of samples: conservative 30% of cohort size
    n = max(5, int(len(cohort_data) * 0.30))
    print(f"  Cohort '{cohort['name']}': {len(cohort_data)} real → generating {n} synthetic")

    try:
        from sdv.metadata import SingleTableMetadata
        from sdv.single_table import GaussianCopulaSynthesizer
        meta = SingleTableMetadata()
        meta.detect_from_dataframe(cohort_data)
        synth = GaussianCopulaSynthesizer(meta)
        synth.fit(cohort_data)
        samples = synth.sample(num_rows=n)
        samples[target_col_name] = cohort['label']
        all_synthetic_parts.append(samples)
    except Exception as e:
        print(f"    [!] Failed for '{cohort['name']}': {e}")

if all_synthetic_parts:
    synthetic_df = pd.concat(all_synthetic_parts, ignore_index=True)
    print(f'\nTotal synthetic samples: {len(synthetic_df)}')
else:
    print('[!] No synthetic samples produced.')
    synthetic_df = pd.DataFrame()

In [ ]:
print('Synthetic data preview:')
print(synthetic_df.head())
print(f'\nSynthetic class distribution:')
print(synthetic_df['is_fraud'].value_counts())

## 4. Synthetic Data Quality Evaluation

In [ ]:
quality_report = evaluate_synthetic_quality(
    real_data=train_full,
    synthetic_data=synthetic_df,
    target_col='is_fraud'
)

print('\nDetailed per-column quality report:')
for col, stats in quality_report['per_column'].items():
    print(f"  {col:10s}  KS={stats['ks_stat']:.4f}  μ-diff={stats['mean_diff']:.4f}  σ-diff={stats['std_diff']:.4f}")

In [ ]:
# --- Distribution comparison plots ---
num_cols = train_full.select_dtypes(include=np.number).columns.tolist()
num_cols = [c for c in num_cols if c in synthetic_df.columns and c != target_col_name][:5]

if num_cols:
    fig, axes = plt.subplots(int(np.ceil(len(num_cols)/3)), 3, figsize=(15, 8))
    if type(axes) == np.ndarray: axes = axes.flatten()
    else: axes = [axes]

    for i, col in enumerate(num_cols):
        real_vals = train_full[col].dropna()
        synt_vals = synthetic_df[col].dropna()
        ks = quality_report['per_column'].get(col, {}).get('ks_stat', np.nan)

        axes[i].hist(real_vals, bins=25, alpha=0.60, color='#3498db', label='Real', edgecolor='white')
        axes[i].hist(synt_vals, bins=25, alpha=0.60, color='#e67e22', label='Synthetic', edgecolor='white')
        axes[i].set_title(f'{col}  (KS={ks:.3f})', fontsize=11)
        axes[i].legend(fontsize=9)

    for j in range(len(num_cols), len(axes)):
        axes[j].axis('off')
        
    plt.suptitle('Real vs Synthetic Distributions', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig('artifacts/synthetic_quality_distributions.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# KS statistic bar chart
ks_data = {col: vals['ks_stat'] for col, vals in quality_report['per_column'].items()}
plt.figure(figsize=(8, 4))
colors = ['#e74c3c' if v > 0.2 else '#f39c12' if v > 0.1 else '#2ecc71' for v in ks_data.values()]
plt.bar(ks_data.keys(), ks_data.values(), color=colors, edgecolor='white')
plt.axhline(0.1, color='orange', linestyle='--', label='Warning (KS=0.10)')
plt.axhline(0.2, color='red',    linestyle='--', label='Danger  (KS=0.20)')
plt.title('KS Statistics per Feature (lower = more similar)', fontweight='bold')
plt.ylabel('KS Statistic')
plt.legend()
plt.tight_layout()
plt.savefig('artifacts/synthetic_ks_statistics.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Save Artefacts

In [ ]:
import os
os.makedirs('artifacts', exist_ok=True)

# Save raw synthetic samples
synthetic_df.to_csv('artifacts/synthetic_samples.csv', index=False)
print(f'synthetic_samples.csv saved  ({len(synthetic_df)} rows)')

# Save the global generator
save_generator(generator, 'artifacts/sdv_copula_model.pkl')

# Save quality report
import json
with open('artifacts/synthetic_quality_report.json', 'w') as f:
    json.dump(quality_report, f, indent=2)
print('synthetic_quality_report.json saved')

print(f'\n=== Quality Summary ===')
print(f'  Mean KS stat : {quality_report["overall_ks_mean"]}')
print(f'  Max  KS stat : {quality_report["overall_ks_max"]}')